# Replication: Iterative Inference in a Chess-Playing Neural Network

This notebook replicates key experiments from the paper investigating how neural networks progressively build understanding across layers using the logit lens technique applied to Leela Chess Zero.

## Key Hypotheses
1. Neural networks perform iterative inference with capability progression occurring in distinct computational phases
2. The inference process shows a three-phase pattern: early rapid gains, middle plateau, late sharp strengthening

In [ ]:
#!/usr/bin/env python3"""Replication: Iterative Inference in a Chess-Playing Neural NetworkThis script replicates key experiments from the paper investigating how neuralnetworks progressively build understanding across layers using the logit lenstechnique applied to Leela Chess Zero.Key Hypotheses:1. Neural networks perform iterative inference with capability progression   occurring in distinct computational phases2. The inference process shows a three-phase pattern: early rapid gains,   middle plateau, late sharp strengthening"""import sysimport os# Setup pathsREPO_ROOT = "/net/scratch2/smallyan/leela_eval"os.chdir(REPO_ROOT)sys.path.insert(0, os.path.join(REPO_ROOT, "src"))import torchimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy.spatial.distance import jensenshannonimport scipy.stats as stimport randomimport jsonfrom pathlib import Path# Set random seeds for reproducibilitySEED = 42random.seed(SEED)np.random.seed(SEED)torch.manual_seed(SEED)# Check devicedevice = "cuda" if torch.cuda.is_available() else "cpu"print(f"Using device: {device}")if torch.cuda.is_available():    torch.cuda.manual_seed(SEED)    torch.backends.cudnn.deterministic = True    torch.backends.cudnn.benchmark = False    print(f"CUDA device: {torch.cuda.get_device_name(0)}")# Create output directoryOUTPUT_DIR = Path(REPO_ROOT) / "evaluation" / "replications" / "figures"OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from leela_interp import Lc0sight, LeelaBoardfrom leela_logit_lens import LeelaLogitLens# Load modelmodel_path = os.path.join(REPO_ROOT, "lc0-original.onnx")print(f"Loading model from: {model_path}")model = Lc0sight(path=model_path, device=device)model.eval()print("Model loaded successfully")print(f"Number of layers: {model.N_LAYERS}")print(f"Model dimension: {model.D_MODEL}")# Initialize logit lenslens = LeelaLogitLens(model)print(f"LeelaLogitLens initialized with {lens.num_layers} layers")print("\n" + "=" * 60)print("PART 2: Demo - Layer-wise Policy Evolution")

In [ ]:
# Create a sample chess position (famous puzzle position)# Using a tactical puzzle position from the demo notebookpuzzle_fen = 'Q1b3k1/5ppp/p3r3/2p1Nn2/2Pq1P2/8/PP1P2PP/R1B2R1K b - - 0 17'board = LeelaBoard.from_fen(puzzle_fen)print(f"Testing on position: {puzzle_fen}")print(f"Board:\n{board}")# Run multi-layer lens analysisresults = lens.multi_layer_lens(    boards=board,    layer_indices=None,  # All layers    return_probs=True,    return_policy_as_dict=True)print(f"\nAnalyzed {len(results[0]['layers'])} layers")# Print top moves for each layerprint("\nTop 3 moves by layer:")print("-" * 50)for layer_idx in sorted(results[0]['layers'].keys()):    policy_dict = results[0]['layers'][layer_idx]['policy_as_dict']    sorted_moves = sorted(policy_dict.items(), key=lambda x: x[1], reverse=True)[:3]    moves_str = ", ".join([f"{m}:{p:.3f}" for m, p in sorted_moves])    layer_name = "Input" if layer_idx == 0 else ("Full Model" if layer_idx == 15 else f"Layer {layer_idx-1}")    print(f"{layer_name:15s}: {moves_str}")print("\n" + "=" * 60)print("PART 3: Policy Distribution Metrics on CCRL Dataset")

In [ ]:
# Sample positions from CCRL datasetfrom leela_logit_lens.tools.sample_positions import sample_unique_positionsccrl_dir = os.path.join(REPO_ROOT, "data", "cclr", "train")print(f"Sampling positions from: {ccrl_dir}")# Sample 100 positions for faster replication (original uses 1000)n_samples = 100print(f"Sampling {n_samples} unique positions...")boards = sample_unique_positions(directory=ccrl_dir, total_samples=n_samples, seed=SEED)print(f"Sampled {len(boards)} positions")# Run multi-layer analysis on all sampled positionsprint("Running multi-layer lens analysis...")results_ccrl = lens.multi_layer_lens(    boards=boards,    output="policy",    return_probs=True,    return_policy_as_dict=True)print(f"Analyzed {len(results_ccrl)} positions")# Helper functions for metric computationdef compute_js_divergence_trajectories(results, model):    """Compute Jensen-Shannon divergence trajectories for all boards."""    layer_indices = sorted(results[0]["layers"].keys())    final_layer_idx = max(layer_indices)    all_trajectories = []    for board_result in results:        board = board_result["board"]        legal_indices, _ = model.legal_moves(board)        legal_indices = torch.tensor(legal_indices, device=model.device)        final_policy = board_result["layers"][final_layer_idx]["policy"]        final_probs = final_policy[legal_indices].cpu().numpy()        final_probs = final_probs / final_probs.sum()        js_trajectory = []        for layer_idx in layer_indices:            layer_policy = board_result["layers"][layer_idx]["policy"]            layer_probs = layer_policy[legal_indices].cpu().numpy()            layer_probs = layer_probs / layer_probs.sum()            js_div = jensenshannon(layer_probs, final_probs, base=2)            js_trajectory.append(js_div)        all_trajectories.append(js_trajectory)    return np.array(all_trajectories)def compute_entropy_trajectories(results, model):    """Compute normalized entropy trajectories for all boards."""    layer_indices = sorted(results[0]["layers"].keys())    all_trajectories = []    for board_result in results:        board = board_result["board"]        legal_indices, _ = model.legal_moves(board)        legal_indices = torch.tensor(legal_indices, device=model.device)        if len(legal_indices) < 2:            continue        num_legal_moves = len(legal_indices)        entropy_trajectory = []        for layer_idx in layer_indices:            layer_policy = board_result["layers"][layer_idx]["policy"]            layer_legal_probs = layer_policy[legal_indices].cpu().numpy()            layer_legal_probs = layer_legal_probs / np.sum(layer_legal_probs)            entropy = -np.sum(layer_legal_probs * np.log2(layer_legal_probs + 1e-12))            max_entropy = np.log2(num_legal_moves)            normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0.0            entropy_trajectory.append(normalized_entropy)        all_trajectories.append(entropy_trajectory)    return np.array(all_trajectories)def compute_tau_trajectories(results, model, top_k=5):    """Compute Kendall's tau trajectories using top-k moves."""    if not results:        return np.array([])    num_layers = len(results[0]["layers"])    layer_indices = sorted(results[0]["layers"].keys())    final_layer_idx = max(layer_indices)    layer_taus = [[] for _ in range(num_layers)]    for board_result in results:        board = board_result["board"]        legal_indices, _ = model.legal_moves(board)        legal_indices = torch.tensor(legal_indices, device=model.device)        if len(legal_indices) < 2:            continue        # Find all moves that appear in top-k of any layer        top_moves_set = set()        for layer_idx in layer_indices:            layer_policy = board_result["layers"][layer_idx]["policy"]            layer_legal_probs = layer_policy[legal_indices]            top_indices = layer_legal_probs.argsort(descending=True)[:top_k]            for idx in top_indices:                top_moves_set.add(idx.item())        top_moves_list = sorted(list(top_moves_set))        if len(top_moves_list) < 2:            continue        # Get final layer ranking        final_policy = board_result["layers"][final_layer_idx]["policy"]        final_legal_probs = final_policy[legal_indices]        final_top_probs = final_legal_probs[top_moves_list]        final_ranking = final_top_probs.argsort(descending=True)        for i, layer_idx in enumerate(layer_indices):            layer_policy = board_result["layers"][layer_idx]["policy"]            layer_legal_probs = layer_policy[legal_indices]            layer_top_probs = layer_legal_probs[top_moves_list]            layer_ranking = layer_top_probs.argsort(descending=True)            final_positions = torch.zeros_like(final_ranking)            layer_positions = torch.zeros_like(layer_ranking)            for rank, move_idx in enumerate(final_ranking):                final_positions[move_idx] = rank            for rank, move_idx in enumerate(layer_ranking):                layer_positions[move_idx] = rank            tau = st.kendalltau(                layer_positions.cpu().numpy(),                final_positions.cpu().numpy(),                variant="b"            ).correlation            if not np.isnan(tau):                layer_taus[i].append(tau)    # Convert to trajectories format    all_trajectories = []    num_valid_boards = len(layer_taus[0]) if layer_taus[0] else 0    for board_idx in range(num_valid_boards):        tau_trajectory = []        for layer_idx in range(num_layers):            if board_idx < len(layer_taus[layer_idx]):                tau_trajectory.append(layer_taus[layer_idx][board_idx])            else:                tau_trajectory.append(0.0)        all_trajectories.append(tau_trajectory)    return np.array(all_trajectories)def compute_top_prediction_trajectories(results, model):    """Compute probability of top-prediction across layers."""    layer_indices = sorted(results[0]["layers"].keys())    final_layer_idx = max(layer_indices)    all_trajectories = []    for board_result in results:        board = board_result["board"]        legal_indices, _ = model.legal_moves(board)        legal_indices = torch.tensor(legal_indices, device=model.device)        if len(legal_indices) < 2:            continue        final_policy = board_result["layers"][final_layer_idx]["policy"]        final_legal_probs = final_policy[legal_indices]        top_move_idx = torch.argmax(final_legal_probs)        prob_trajectory = []        for layer_idx in layer_indices:            layer_policy = board_result["layers"][layer_idx]["policy"]            layer_legal_probs = layer_policy[legal_indices]            top_move_prob = layer_legal_probs[top_move_idx].cpu().numpy()            prob_trajectory.append(float(top_move_prob))        all_trajectories.append(prob_trajectory)    return np.array(all_trajectories)# Compute metricsprint("\nComputing metrics...")print("Computing JS divergence...")js_data = compute_js_divergence_trajectories(results_ccrl, model)print(f"JS divergence shape: {js_data.shape}")print("Computing entropy...")entropy_data = compute_entropy_trajectories(results_ccrl, model)print(f"Entropy shape: {entropy_data.shape}")print("Computing Kendall's tau...")tau_data = compute_tau_trajectories(results_ccrl, model, top_k=5)print(f"Tau shape: {tau_data.shape}")print("Computing top prediction probability...")top_pred_data = compute_top_prediction_trajectories(results_ccrl, model)print(f"Top prediction shape: {top_pred_data.shape}")# Plotting functiondef plot_metric(data_array, ylabel, save_path, title=None, ylim=(None, 1)):    """Plot metric with median and percentile bands."""    median_vals = np.median(data_array, axis=0)    q25 = np.percentile(data_array, 25, axis=0)    q75 = np.percentile(data_array, 75, axis=0)    q10 = np.percentile(data_array, 5, axis=0)    q90 = np.percentile(data_array, 95, axis=0)    num_layers = len(median_vals)    # Create x-tick labels    x_tick_labels = ["Input"] + [str(i) for i in range(num_layers-2)] + ["Final"]    fig, ax = plt.subplots(figsize=(10, 6))    # Plot median    ax.plot(range(num_layers), median_vals, 'b-', linewidth=2, label='Median')    # Plot percentile bands    ax.fill_between(range(num_layers), q10, q90, alpha=0.2, color='blue', label='5th-95th percentile')    ax.fill_between(range(num_layers), q25, q75, alpha=0.4, color='blue', label='25th-75th percentile')    # Add phase shading    ax.axvspan(0, 6, facecolor='gray', alpha=0.1, label='Early phase')    ax.axvspan(11, 15, facecolor='gray', alpha=0.1, label='Late phase')    ax.set_xlabel('Layer')    ax.set_ylabel(ylabel)    if title:        ax.set_title(title)    ax.set_xlim(0, num_layers - 1)    if ylim[0] is not None or ylim[1] is not None:        ax.set_ylim(ylim)    ax.set_xticks(range(num_layers))    ax.set_xticklabels(x_tick_labels, rotation=45)    ax.grid(True, alpha=0.3)    ax.legend(loc='best')    plt.tight_layout()    plt.savefig(save_path, dpi=150)    plt.close()    print(f"Saved: {save_path}")    return median_valsprint("\nGenerating plots...")# Plot JS divergencejs_median = plot_metric(    js_data,    ylabel="Jensen-Shannon Divergence",    save_path=OUTPUT_DIR / "js_divergence.png",    title="JS Divergence from Final Layer Policy")# Plot entropyentropy_median = plot_metric(    entropy_data,    ylabel="Normalized Entropy",    save_path=OUTPUT_DIR / "entropy.png",    title="Normalized Policy Entropy",    ylim=(0, 1))# Plot tautau_median = plot_metric(    tau_data,    ylabel="Kendall τ (Ranking Correlation)",    save_path=OUTPUT_DIR / "tau_correlation.png",    title="Kendall τ Ranking Correlation (Top-5 Moves)",    ylim=(-0.5, 1.05))# Plot top predictiontop_pred_median = plot_metric(    top_pred_data,    ylabel="Probability",    save_path=OUTPUT_DIR / "top_prediction.png",    title="Probability of Final Top Move",    ylim=(0, 1))print("\n" + "=" * 60)print("PART 4: Results Summary")

In [ ]:
# Summary statisticsprint("\nMetric Statistics by Layer:")print("-" * 60)print(f"{'Metric':<25} {'Early (L0-5)':<15} {'Middle (L6-10)':<15} {'Late (L11-14)':<15}")print("-" * 60)early_mask = slice(1, 7)  # Layers 0-5middle_mask = slice(7, 12)  # Layers 6-10late_mask = slice(12, 16)  # Layers 11-14print(f"{'JS Divergence (mean)':<25} {js_median[early_mask].mean():.4f}          {js_median[middle_mask].mean():.4f}          {js_median[late_mask].mean():.4f}")print(f"{'Entropy (mean)':<25} {entropy_median[early_mask].mean():.4f}          {entropy_median[middle_mask].mean():.4f}          {entropy_median[late_mask].mean():.4f}")print(f"{'Tau (mean)':<25} {tau_median[early_mask].mean():.4f}          {tau_median[middle_mask].mean():.4f}          {tau_median[late_mask].mean():.4f}")print(f"{'Top Pred Prob (mean)':<25} {top_pred_median[early_mask].mean():.4f}          {top_pred_median[middle_mask].mean():.4f}          {top_pred_median[late_mask].mean():.4f}")print("\n" + "=" * 60)print("PART 5: Three-Phase Pattern Verification")

In [ ]:
# Verify the three-phase pattern# According to the paper:# - Early phase (L0-5): Rapid change# - Middle phase (L6-10): Plateau# - Late phase (L11-14): Sharp strengthening# Calculate phase-wise changes in Kendall tau (key metric)tau_early_change = tau_median[6] - tau_median[0]  # Change in early phasetau_middle_change = tau_median[11] - tau_median[6]  # Change in middle phasetau_late_change = tau_median[15] - tau_median[11]  # Change in late phaseprint("\nKendall τ Changes by Phase:")print(f"  Early phase (Input→L5):  {tau_early_change:+.4f}")print(f"  Middle phase (L5→L10):   {tau_middle_change:+.4f}")print(f"  Late phase (L10→Final):  {tau_late_change:+.4f}")# Verify patternearly_rapid = tau_early_change > 0middle_plateau = abs(tau_middle_change) < abs(tau_early_change) * 0.5late_sharp = tau_late_change > tau_middle_changeprint("\nThree-Phase Pattern Verification:")print(f"  Early rapid gain: {'✓ VERIFIED' if early_rapid else '✗ NOT VERIFIED'}")print(f"  Middle plateau:   {'✓ VERIFIED' if middle_plateau else '✗ NOT VERIFIED'}")print(f"  Late sharpening:  {'✓ VERIFIED' if late_sharp else '✗ NOT VERIFIED'}")# Save results summaryresults_summary = {    "experiment": "Iterative Inference Replication",    "n_positions": n_samples,    "seed": SEED,    "device": device,    "metrics": {        "js_divergence": {            "early_mean": float(js_median[early_mask].mean()),            "middle_mean": float(js_median[middle_mask].mean()),            "late_mean": float(js_median[late_mask].mean()),        },        "entropy": {            "early_mean": float(entropy_median[early_mask].mean()),            "middle_mean": float(entropy_median[middle_mask].mean()),            "late_mean": float(entropy_median[late_mask].mean()),        },        "kendall_tau": {            "early_mean": float(tau_median[early_mask].mean()),            "middle_mean": float(tau_median[middle_mask].mean()),            "late_mean": float(tau_median[late_mask].mean()),            "early_change": float(tau_early_change),            "middle_change": float(tau_middle_change),            "late_change": float(tau_late_change),        },        "top_prediction": {            "early_mean": float(top_pred_median[early_mask].mean()),            "middle_mean": float(top_pred_median[middle_mask].mean()),            "late_mean": float(top_pred_median[late_mask].mean()),        }    },    "three_phase_verification": {        "early_rapid_gain": bool(early_rapid),        "middle_plateau": bool(middle_plateau),        "late_sharpening": bool(late_sharp),    }}results_path = OUTPUT_DIR.parent / "replication_results.json"with open(results_path, 'w') as f:    json.dump(results_summary, f, indent=2)print(f"\nResults saved to: {results_path}")print("\n" + "=" * 60)print("REPLICATION COMPLETE")